# Trening 5 algorytmow detekcji anomalii (Colab)

Notebook trenuje warstwe 3 detektora (`detection/ml.py`) na **Drone Telemetry Tampering Dataset v2**
uzywajac 5 roznych algorytmow i porownuje ich skutecznosc.

**Wymaga:** plik `kaggle.json` z tokenem Kaggle (Account -> Create New API Token).

**Wynik:** 5 pickle'i modeli w `models/` + tabele porownawcze + wykresy ROC.

## 1. Konfiguracja srodowiska

In [ ]:
!pip install -q kaggle xgboost

from google.colab import files
import os

# Wgraj kaggle.json (Account -> Create New API Token na kaggle.com)
if not os.path.exists('/root/.kaggle/kaggle.json'):
    files.upload()
    !mkdir -p /root/.kaggle && mv kaggle.json /root/.kaggle/ && chmod 600 /root/.kaggle/kaggle.json

## 2. Klonowanie repo projektu i pobranie datasetu

In [ ]:
# Opcja A: klonowanie z GitHub (zamien URL na wlasny)
# !git clone https://github.com/<user>/drone_anomaly.git
# %cd drone_anomaly

# Opcja B: wrzucenie kodu recznie - uplaoduj plik .zip i rozpakuj:
# from google.colab import files; files.upload()
# !unzip -q drone_anomaly.zip

# Pobierz dataset
!mkdir -p data models
!kaggle datasets download -d rasikaekanayakadevlk/drone-telemetry-tampering-dataset-v2 -p data --unzip
!ls -lh data/

In [ ]:
# Wskaz wlasciwy plik CSV (nazwa moze sie roznic od domyslnej)
import glob
csvs = glob.glob('data/*.csv')
print(csvs)
# Jesli nazwa jest inna - przemianuj do oczekiwanej nazwy:
# !mv data/<actual_name>.csv data/drone_telemetry_v2.csv

## 3. Wczytanie i podsumowanie datasetu

In [ ]:
import sys
sys.path.insert(0, '.')

from data.loader import load_dataset, split_by_replicate, summarize

df = load_dataset('data/drone_telemetry_v2.csv')
summarize(df)
train, test = split_by_replicate(df)
print(f'\nTrain: {len(train):,}  Test: {len(test):,}')

## 4. Trening 5 algorytmow + ewaluacja

In [ ]:
import time
import pandas as pd

from detection.rules import detect_threshold_violations
from detection.statistical import detect_sudden_changes
from detection.ml import AnomalyDetector
from evaluation.metrics import compute_layer_metrics

# Wstepnie: warstwy 1 i 2 na zbiorze testowym (raz, nie zaleza od algorytmu W3)
print('Warstwa 1 + 2 na test...')
test_base = detect_threshold_violations(test)
test_base = detect_sudden_changes(test_base)

ALGORITHMS = ['isolation_forest', 'one_class_svm', 'lof', 'random_forest', 'xgboost']

results = {}
predictions = {}

for algo in ALGORITHMS:
    print(f'\n=== {algo} ===')
    t0 = time.time()
    try:
        det = AnomalyDetector(algorithm=algo).fit(train)
        out = det.predict(test_base)
        det.save(f'models/{algo}.pkl')
        m = compute_layer_metrics(out, 'alert_ml')
        results[algo] = {**m, 'time_s': round(time.time() - t0, 1)}
        predictions[algo] = out
        print(f'  P={m["precision"]:.3f}  R={m["recall"]:.3f}  F1={m["f1"]:.3f}  '
              f'(czas: {results[algo]["time_s"]}s)')
    except Exception as e:
        print(f'  BLAD: {e}')
        results[algo] = None

summary = pd.DataFrame({k: v for k, v in results.items() if v}).T
print('\nPodsumowanie:')
summary

## 5. Krzywe ROC - porownanie algorytmow

In [ ]:
from evaluation.metrics import plot_roc_curves

# Dla kazdego algorytmu dodajemy score do jednego DF testowego
merged = test_base.copy()
score_columns = {}
UNSUP = {'isolation_forest', 'one_class_svm', 'lof'}
for algo, pred in predictions.items():
    col = f'score_{algo}'
    merged[col] = pred['ml_score'].values
    direction = 'low' if algo in UNSUP else 'high'
    score_columns[algo] = (col, direction)

plot_roc_curves(merged, score_columns,
                binary_layers={'alert_threshold': 'W1', 'alert_change': 'W2'},
                output_path='data/roc_curves_all_algos.png')

## 6. Pobranie modeli i wynikow

In [ ]:
!zip -r results.zip models/ data/*.png data/*.csv
from google.colab import files
files.download('results.zip')